# AIMO3 Continuous Adaptive Solver (Feb 5)

## Strategy: Continuous Uncertainty-Based Resampling

Unlike a two-phase approach, this solver continuously monitors uncertainty and allocates extra attempts throughout the 8-hour window:

1. **Initial pass**: Solve all 50 problems with base attempts (6 each)
2. **Continuous loop**: While time remains:
   - Find the most uncertain problem (lowest consensus, highest entropy)
   - Run extra attempts on it
   - Update its answer and uncertainty metrics
   - Repeat until time runs out
3. **Finalize**: Only prepare final answers when time limit approaches

Based on feb3 (scored 40/50) with continuous adaptive resource allocation.

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess, gc, re, math, time, queue, threading, contextlib, heapq

In [ ]:
def set_env(archive, tmp):
    if not os.path.exists(tmp):
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', tmp], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', f'{tmp}/wheels',
                    'unsloth', 'trl', 'vllm', 'openai_harmony'], check=True)

set_env('/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false'), ('TRITON_PTXAS_PATH', '/usr/local/cuda/bin/ptxas'),
             ('TIKTOKEN_ENCODINGS_BASE', '/kaggle/tmp/setup/tiktoken_encodings')]:
    os.environ[k] = v

In [ ]:
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import pandas as pd, polars as pl
from openai import OpenAI
from openai_harmony import (HarmonyEncodingName, load_harmony_encoding, SystemContent, ReasoningEffort,
                             ToolNamespaceConfig, Author, Message, Role, TextContent, Conversation)
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

## Configuration

**Key parameters for continuous adaptive approach:**
- `time_limit = 28800` — Full 8 hours (competition allows 9h, we use 8h for safety)
- `finalize_buffer = 300` — Stop resampling 5 min before deadline
- `base_attempts = 6` — Initial attempts per problem
- `resample_batch = 2` — Extra attempts per resample cycle

In [ ]:
class CFG:
    # Prompt - single proven reasoning prompt
    system_prompt = ('You are a world-class International Mathematical Olympiad (IMO) competitor. '
                    'The final answer must be a non-negative integer between 0 and 99999. '
                    'You must place the final integer answer inside \\boxed{}.')
    tool_prompt = ('Use this tool to execute Python code. The environment is a stateful Jupyter notebook. '
                  'You must use print() to output results.')
    preference_prompt = 'You have access to `math`, `numpy` and `sympy` to solve the problem.'

    # Model - same as feb3
    served_model_name, model_path = 'gpt-oss', '/kaggle/input/gpt-oss-120b/transformers/default/1'
    kv_cache_dtype, dtype = 'fp8_e4m3', 'auto'

    # TIMING — Use full 8 hours
    time_limit = 28800            # 8 hours total (competition allows 9h)
    finalize_buffer = 300         # Stop resampling 5 min before deadline
    problem_timeout = 420         # Max 7 min per attempt batch
    server_timeout = 180
    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    # Generation - same as feb3
    stream_interval, context_tokens, buffer_tokens, search_tokens = 200, 65536, 512, 32
    top_logprobs, batch_size = 5, 256
    gpu_memory_utilization, temperature, min_p, seed = 0.96, 1.0, 0.02, 42

    # ADAPTIVE PARAMETERS
    workers = 8                   # Parallel workers
    turns = 128                   # Max turns per attempt
    base_attempts = 6             # Initial attempts per problem
    early_stop = 4                # Stop early if N answers agree
    resample_batch = 2            # Extra attempts per resample cycle
    max_attempts_per_problem = 20 # Cap to avoid infinite resampling

    # Uncertainty thresholds
    confident_consensus = 4       # Problem is confident if top answer has >= this many votes
    confident_entropy = 2.0       # Problem is confident if min entropy <= this

    # Entropy-Gated Consensus (same as feb3)
    entropy_threshold = 5.0
    min_consensus = 2

print(f"Time limit: {CFG.time_limit}s ({CFG.time_limit/3600:.1f}h)")
print(f"Base attempts: {CFG.base_attempts}, Resample batch: {CFG.resample_batch}")
print(f"Confident if: consensus >= {CFG.confident_consensus} OR min_entropy <= {CFG.confident_entropy}")

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:
    def get_system_content(self, prompt, tool_cfg):
        return SystemContent.new().with_model_identity(prompt).with_reasoning_effort(
            reasoning_effort=ReasoningEffort.HIGH).with_tools(tool_cfg)

    def apply_chat_template(self, sys_prompt, usr_prompt, tool_cfg):
        return [Message.from_role_and_content(Role.SYSTEM, self.get_system_content(sys_prompt, tool_cfg)),
                Message.from_role_and_content(Role.USER, usr_prompt)]

In [ ]:
class AIMO3Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout):
        self._default_timeout, self._owns_kernel, self._client, self._km = timeout, False, None, None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
                   'JUPYTER_PLATFORM_DIRS': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute('import math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

    def _format_error(self, tb):
        return ''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb
                      if 'File "' not in f or 'ipython-input' in f)

    def execute(self, code, timeout=None):
        effective_timeout = timeout or self._default_timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                stderr.append(self._format_error(c.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[WARN] No output. Use print() to see results.')

    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()

    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

    def __del__(self):
        self.close()

In [ ]:
class AIMO3Tool:
    def __init__(self, timeout, prompt, sandbox=None):
        self._local_jupyter_timeout, self._tool_prompt, self._jupyter_session = timeout, prompt, sandbox
        self._owns_session, self._execution_lock, self._init_lock = sandbox is None, threading.Lock(), threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if any(x in last for x in ['print', 'import']) or not last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)

    @property
    def instruction(self): return self._tool_prompt

    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self.instruction, tools=[])

    def _make_response(self, output, channel=None):
        msg = Message(author=Author(role=Role.TOOL, name='python'),
                     content=[TextContent(text=output)]).with_recipient('assistant')
        return msg.with_channel(channel) if channel else msg

    def process_sync_plus(self, message):
        self._ensure_session()
        final_script = self._ensure_last_print(message.content[0].text)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

## AIMO3 Continuous Adaptive Solver

Key features:
1. `uncertainty_queue` — Priority queue of (uncertainty_score, problem_id) for resampling
2. `_compute_uncertainty()` — Score combining consensus and entropy
3. `_resample_uncertain()` — Run extra attempts on most uncertain problem
4. `run_adaptive_loop()` — Continuous loop until time runs out

In [ ]:
class AIMO3ContinuousAdaptiveSolver:
    def __init__(self, cfg, port=8000):
        self.cfg, self.port = cfg, port
        self.base_url, self.api_key = f'http://0.0.0.0:{port}/v1', 'sk-local'
        self.template, self.encoding = AIMO3Template(), load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
        self._wait_for_server()
        self._initialize_kernels()
        self.notebook_start_time = time.time()

        # ADAPTIVE: Store all problem data
        self.problem_data = {}       # problem_id -> {'question': str, 'results': list, 'answer': int, 'uncertainty': float}
        self.problem_order = []      # Track order for output
        self.initial_pass_done = False
        self.resample_count = defaultdict(int)  # Track resamples per problem

    def _preload_model_weights(self):
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start, files, total = time.time(), [], 0
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp):
                    files.append(fp)
                    total += os.path.getsize(fp)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            list(ex.map(lambda p: open(p, 'rb').read(), files))
        print(f'Processed {len(files)} files ({total/1e9:.2f} GB) in {time.time()-start:.2f} seconds.\n')

    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--seed', str(self.cfg.seed),
               '--model', self.cfg.model_path, '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1', '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), '--host', '0.0.0.0',
               '--port', str(self.port), '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype,
               '--max-model-len', str(self.cfg.context_tokens), '--stream-interval', str(self.cfg.stream_interval),
               '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching']
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if (rc := self.server_process.poll()) is not None:
                self.log_file.flush()
                raise RuntimeError(f'Server died with code {rc}. Full logs:\n{open("vllm_server.log").read()}\n')
            try:
                self.client.models.list()
                print(f'Server is ready (took {time.time()-start:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self):
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start = time.time()
        self.sandbox_pool = queue.Queue()
        for i in range(self.cfg.workers):
            for attempt in range(3):
                try:
                    sandbox = AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
                    self.sandbox_pool.put(sandbox)
                    print(f'  Kernel {i+1}/{self.cfg.workers} ready')
                    break
                except Exception as e:
                    if attempt < 2:
                        print(f'  Kernel {i+1} attempt {attempt+1} failed, retrying...')
                        time.sleep(1)
                    else:
                        print(f'  Kernel {i+1} failed after 3 attempts: {e}')
        print(f'Kernels initialized in {time.time()-start:.2f} seconds ({self.sandbox_pool.qsize()} ready).\n')

    def _time_remaining(self):
        return self.cfg.time_limit - (time.time() - self.notebook_start_time)

    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except ValueError: pass
        return None

    def _compute_mean_entropy(self, logprobs):
        if not logprobs: return float('inf')
        total, count = 0.0, 0
        for top_lp in logprobs:
            if isinstance(top_lp, dict) and top_lp:
                ent = sum(-math.exp(lp)*math.log2(math.exp(lp)) for lp in top_lp.values() if math.exp(lp) > 0)
                total += ent
                count += 1
        return total/count if count else float('inf')

    def _process_attempt(self, problem, sys_prompt, idx, stop_evt, deadline):
        if stop_evt.is_set() or time.time() > deadline:
            return {'Attempt': idx+1, 'Answer': None, 'Python Calls': 0, 'Python Errors': 0,
                   'Response Length': 0, 'Entropy': float('inf')}
        local_tool, sandbox, py_calls, py_errs, total_toks, ans, logprobs = None, None, 0, 0, 0, None, []
        seed = int(math.pow(self.cfg.seed + idx, 2))
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(self.cfg.jupyter_timeout, self.cfg.tool_prompt, sandbox)
            conv = Conversation.from_messages(self.template.apply_chat_template(
                sys_prompt, problem, local_tool.tool_config))
            for _ in range(self.cfg.turns):
                if stop_evt.is_set() or time.time() > deadline: break
                prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
                if (max_toks := self.cfg.context_tokens - len(prompt_ids)) < self.cfg.buffer_tokens: break
                stream = self.client.completions.create(model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature, logprobs=self.cfg.top_logprobs, max_tokens=max_toks,
                    prompt=prompt_ids, seed=seed, stream=True, extra_body={
                        'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True})
                try:
                    tok_buf, txt_chunks = [], []
                    for chunk in stream:
                        if stop_evt.is_set() or time.time() > deadline: break
                        if new_toks := chunk.choices[0].token_ids:
                            tok_buf.extend(new_toks)
                            total_toks += len(new_toks)
                            txt_chunks.append(chunk.choices[0].text)
                            if (clp := chunk.choices[0].logprobs) and clp.top_logprobs:
                                logprobs.extend(clp.top_logprobs)
                        if '}' in chunk.choices[0].text and (ans := self._scan_for_answer(
                            ''.join(txt_chunks[-self.cfg.search_tokens:]))):
                            break
                finally:
                    stream.close()
                if ans or not tok_buf: break
                new_msgs = self.encoding.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs)
                last = new_msgs[-1]
                if last.channel == 'final':
                    ans = self._scan_for_answer(last.content[0].text)
                    break
                if last.recipient == 'python':
                    py_calls += 1
                    resp = local_tool.process_sync_plus(last)
                    if any(x in (txt := resp[0].content[0].text) for x in ['[ERROR]', 'Traceback', 'Error:']):
                        py_errs += 1
                    conv.messages.extend(resp)
        except Exception: py_errs += 1
        finally:
            if sandbox:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
        return {'Attempt': idx+1, 'Response Length': total_toks, 'Python Calls': py_calls,
               'Python Errors': py_errs, 'Entropy': self._compute_mean_entropy(logprobs), 'Answer': ans}

    def _select_answer_gated(self, results):
        """Entropy-Gated Consensus selection. Returns (answer, metrics)."""
        valid_results = [r for r in results if r['Answer'] is not None]
        if not valid_results:
            return 0, {'consensus': 0, 'min_entropy': float('inf')}

        # Filter by entropy threshold
        confident_results = [r for r in valid_results if r['Entropy'] < self.cfg.entropy_threshold]

        # Count votes, require consensus
        if confident_results:
            confident_votes = Counter(r['Answer'] for r in confident_results)
            candidates = {ans: cnt for ans, cnt in confident_votes.items() if cnt >= self.cfg.min_consensus}
        else:
            candidates = {}

        # Metrics for uncertainty calculation
        all_votes = Counter(r['Answer'] for r in valid_results)
        top_consensus = all_votes.most_common(1)[0][1] if all_votes else 0
        min_entropy = min(r['Entropy'] for r in valid_results) if valid_results else float('inf')

        # Entropy-weighted scoring
        if candidates:
            scores = defaultdict(float)
            for r in confident_results:
                if r['Answer'] in candidates:
                    scores[r['Answer']] += 1.0 / max(r['Entropy'], 0.1)
            final = max(scores, key=scores.get)
        else:
            # Fallback to majority
            final = all_votes.most_common(1)[0][0]

        return final, {'consensus': top_consensus, 'min_entropy': min_entropy}

    def _compute_uncertainty(self, metrics):
        """Compute uncertainty score (higher = more uncertain, needs resampling)."""
        # Normalize consensus (0 = max consensus, 1 = no consensus)
        consensus_score = 1.0 - min(metrics['consensus'] / self.cfg.confident_consensus, 1.0)

        # Normalize entropy (0 = very confident, 1 = very uncertain)
        entropy_score = min(metrics['min_entropy'] / 10.0, 1.0)  # Cap at 10.0

        # Combined score (weight consensus more heavily)
        return 0.7 * consensus_score + 0.3 * entropy_score

    def _is_confident(self, metrics):
        """Check if we're confident enough to not resample."""
        return (metrics['consensus'] >= self.cfg.confident_consensus or
                metrics['min_entropy'] <= self.cfg.confident_entropy)

    def solve_problem_initial(self, problem_id, problem):
        """Initial solve with base attempts."""
        print(f'\n[INITIAL] Problem {problem_id}: {problem[:80]}...')
        user_input = f'{problem} {self.cfg.preference_prompt}'
        deadline = time.time() + self.cfg.problem_timeout

        results, valid, stop_evt = [], [], threading.Event()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            futures = [ex.submit(self._process_attempt, user_input, self.cfg.system_prompt, i, stop_evt, deadline)
                      for i in range(self.cfg.base_attempts)]
            for future in as_completed(futures):
                try:
                    if (r := future.result())['Answer'] is not None:
                        valid.append(r['Answer'])
                    results.append(r)
                    if (cnts := Counter(valid).most_common(1)) and cnts[0][1] >= self.cfg.early_stop:
                        stop_evt.set()
                        for f in futures: f.cancel()
                        break
                except Exception as exc:
                    print(f'Future failed: {exc}')

        answer, metrics = self._select_answer_gated(results) if valid else (0, {'consensus': 0, 'min_entropy': float('inf')})
        uncertainty = self._compute_uncertainty(metrics)

        # Store problem data
        self.problem_data[problem_id] = {
            'question': user_input,
            'results': results,
            'answer': answer,
            'metrics': metrics,
            'uncertainty': uncertainty
        }
        self.problem_order.append(problem_id)

        confident = "[CONFIDENT]" if self._is_confident(metrics) else "[UNCERTAIN]"
        print(f'{confident} Answer: {answer} | Consensus: {metrics["consensus"]} | Entropy: {metrics["min_entropy"]:.2f} | Uncertainty: {uncertainty:.3f}')

        if results:
            df = pd.DataFrame(results)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df)

        return answer

    def _get_most_uncertain_problem(self):
        """Find the problem with highest uncertainty that can still be resampled."""
        candidates = [
            (data['uncertainty'], pid)
            for pid, data in self.problem_data.items()
            if (not self._is_confident(data['metrics']) and
                len(data['results']) < self.cfg.max_attempts_per_problem)
        ]
        if not candidates:
            return None
        # Return problem with highest uncertainty
        return max(candidates, key=lambda x: x[0])[1]

    def _resample_problem(self, problem_id):
        """Run extra attempts on a specific problem."""
        data = self.problem_data[problem_id]
        start_idx = len(data['results'])
        self.resample_count[problem_id] += 1

        print(f'\n[RESAMPLE #{self.resample_count[problem_id]}] Problem {problem_id} (uncertainty: {data["uncertainty"]:.3f})')
        print(f'  Current: {len(data["results"])} attempts, answer={data["answer"]}, consensus={data["metrics"]["consensus"]}')

        deadline = time.time() + self.cfg.problem_timeout
        new_results, stop_evt = [], threading.Event()

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            futures = [ex.submit(self._process_attempt, data['question'], self.cfg.system_prompt,
                                start_idx + i, stop_evt, deadline)
                      for i in range(self.cfg.resample_batch)]
            for future in as_completed(futures):
                try:
                    new_results.append(future.result())
                except Exception as exc:
                    print(f'Future failed: {exc}')

        # Merge and re-select
        all_results = data['results'] + new_results
        new_answer, new_metrics = self._select_answer_gated(all_results)
        new_uncertainty = self._compute_uncertainty(new_metrics)

        # Update stored data
        old_answer = data['answer']
        self.problem_data[problem_id]['results'] = all_results
        self.problem_data[problem_id]['answer'] = new_answer
        self.problem_data[problem_id]['metrics'] = new_metrics
        self.problem_data[problem_id]['uncertainty'] = new_uncertainty

        changed = "CHANGED" if new_answer != old_answer else "same"
        confident = "[NOW CONFIDENT]" if self._is_confident(new_metrics) else ""
        print(f'  Result: {old_answer} -> {new_answer} ({changed}) | Consensus: {new_metrics["consensus"]} | Uncertainty: {new_uncertainty:.3f} {confident}')

    def run_adaptive_loop(self):
        """Continuously resample uncertain problems until time runs out."""
        print(f'\n{"="*70}')
        print('ADAPTIVE RESAMPLING LOOP')
        print(f'{"="*70}')

        loop_count = 0
        while self._time_remaining() > self.cfg.finalize_buffer:
            loop_count += 1
            time_left = self._time_remaining()

            # Find most uncertain problem
            problem_id = self._get_most_uncertain_problem()
            if problem_id is None:
                print(f'\n[Loop {loop_count}] All problems are confident or at max attempts. Stopping.')
                break

            print(f'\n[Loop {loop_count}] Time remaining: {time_left:.0f}s ({time_left/60:.1f}min)')

            # Resample
            self._resample_problem(problem_id)

            # Brief status
            uncertain_count = sum(1 for d in self.problem_data.values() if not self._is_confident(d['metrics']))
            print(f'  Status: {uncertain_count} problems still uncertain')

        print(f'\n{"="*70}')
        print(f'ADAPTIVE LOOP COMPLETE after {loop_count} iterations')
        print(f'Time remaining: {self._time_remaining():.0f}s')
        print(f'{"="*70}')

        # Final summary
        self._print_final_summary()

    def _print_final_summary(self):
        """Print summary of all problems and their final states."""
        print('\nFINAL PROBLEM SUMMARY:')
        print(f'{"ID":<12} {"Answer":>8} {"Attempts":>10} {"Consensus":>10} {"Entropy":>10} {"Resamples":>10} {"Status":<12}')
        print('-' * 75)
        for pid in self.problem_order:
            d = self.problem_data[pid]
            status = "confident" if self._is_confident(d['metrics']) else "uncertain"
            print(f'{pid:<12} {d["answer"]:>8} {len(d["results"]):>10} {d["metrics"]["consensus"]:>10} {d["metrics"]["min_entropy"]:>10.2f} {self.resample_count[pid]:>10} {status:<12}')

    def get_final_answer(self, problem_id):
        """Get the final answer for a problem."""
        if problem_id in self.problem_data:
            return self.problem_data[problem_id]['answer']
        return 0

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'): self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                with contextlib.suppress(Exception): self.sandbox_pool.get_nowait().close()

In [ ]:
solver = AIMO3ContinuousAdaptiveSolver(CFG)

## Prediction Flow

1. Each `predict()` call runs initial attempts and stores results
2. After the last problem (50th), trigger the adaptive loop
3. The adaptive loop continuously resamples until time runs out
4. Final answers are retrieved from stored data

In [ ]:
problem_count = [0]
adaptive_done = [False]

def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    gc.disable()

    problem_id = id_.item(0)
    problem_text = question.item(0)

    # Initial solve
    solver.solve_problem_initial(problem_id, problem_text)
    problem_count[0] += 1

    # After last problem, run adaptive loop
    if not adaptive_done[0] and problem_count[0] >= 50:
        print(f'\n*** All {problem_count[0]} problems solved. Starting adaptive resampling loop. ***')
        solver.initial_pass_done = True
        solver.run_adaptive_loop()
        adaptive_done[0] = True

    # Get final answer (may have been updated by adaptive loop)
    final_answer = solver.get_final_answer(problem_id)

    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': problem_id, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',))